# FasalSetu — Phase 3: RAPIDS GPU Forecasting (Google Colab)

**IMPORTANT: Run this notebook in Google Colab with a T4 GPU.**

This notebook reads the historical 7-day, 30-day moving averages and risk scores from BigQuery,
trains an XGBoost model using RAPIDS `cuDF` and `cuML` to forecast future prices, and writes
the predictions back into BigQuery (`fact_forecast`).

## 1. Install Dependencies & Authenticate
Install RAPIDS (cuDF/cuML) and BigQuery dependencies.

In [ ]:
!pip install -q cudf-cu12 cuml-cu12 --extra-index-url=https://pypi.nvidia.com
!pip install -q google-cloud-bigquery pandas pyarrow db-dtypes xgboost

from google.colab import auth
auth.authenticate_user()
print("✅ Authenticated successfully.")

## 2. Import Libraries

In [ ]:
import cudf
import cuml
from cuml.ensemble import RandomForestRegressor
import xgboost as xgb
import pandas as pd
from google.cloud import bigquery
import datetime
import numpy as np

PROJECT_ID = "fasalsetu-501307"
DATASET_ID = "mandi_data"

## 3. Read Data from BigQuery (agg_risk_score)

In [ ]:
print("📥 Fetching historical data and risk scores from BigQuery...")
client = bigquery.Client(project=PROJECT_ID)

query = f"""
    SELECT 
        mandi_id,
        commodity_id,
        date,
        avg_price,
        ma_7d,
        ma_30d,
        volatility_30d,
        risk_score
    FROM `{PROJECT_ID}.{DATASET_ID}.agg_risk_score`
    ORDER BY mandi_id, commodity_id, date
"""

# Download to Pandas first, then convert to RAPIDS GPU DataFrame (cuDF)
df_pd = client.query(query).to_dataframe()
df_pd['date'] = pd.to_datetime(df_pd['date'])
df_pd.dropna(inplace=True)

df = cudf.DataFrame.from_pandas(df_pd)
print(f"✅ Loaded {len(df)} rows into GPU memory (cuDF).")
df.head()

## 4. Feature Engineering
Create lag features and prepare the target variable (future price).

In [ ]:
# We want to predict the price 7 days into the future.
print("🧠 Engineering features...")

# Sort by date
df = df.sort_values(by=['mandi_id', 'commodity_id', 'date'])

# We need to shift the target variable (-7 days) to predict future price
# cuDF supports shift per group using groupby
df['target_price_7d'] = df.groupby(['mandi_id', 'commodity_id'])['avg_price'].shift(-7)

# Drop rows where target is null (the last 7 days of our dataset)
df = df.dropna()

# Extract date features (cuDF datetime extraction)
df['month'] = df['date'].dt.month
df['day_of_week'] = df['date'].dt.weekday

# Convert categorical IDs to numerical codes for XGBoost
df['mandi_code'] = df['mandi_id'].astype('category').cat.codes
df['commodity_code'] = df['commodity_id'].astype('category').cat.codes

features = ['mandi_code', 'commodity_code', 'month', 'day_of_week', 'avg_price', 'ma_7d', 'ma_30d', 'volatility_30d', 'risk_score']
target = 'target_price_7d'

print("✅ Features generated.")

## 5. Train RAPIDS XGBoost Model on GPU

In [ ]:
print("🚀 Training XGBoost model on GPU...")

# Split into train/test based on time
train_date_limit = df['date'].max() - pd.Timedelta(days=30)
train_df = df[df['date'] <= train_date_limit]
test_df = df[df['date'] > train_date_limit]

X_train = train_df[features]
y_train = train_df[target]
X_test = test_df[features]
y_test = test_df[target]

# Train GPU XGBoost
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

params = {
    'tree_method': 'hist',
    'device': 'cuda', # Use GPU
    'objective': 'reg:squarederror',
    'max_depth': 8,
    'learning_rate': 0.1
}

model = xgb.train(params, dtrain, num_boost_round=100, evals=[(dtest, 'test')])
print("✅ Model trained successfully on GPU.")

## 6. Generate Forecasts & Write to BigQuery (`fact_forecast`)

In [ ]:
# Now we generate forecasts for the LATEST data available (the next 7 days)
print("🔮 Generating forecasts...")

latest_data = df_pd.groupby(['mandi_id', 'commodity_id']).tail(1).copy()
latest_cudf = cudf.DataFrame.from_pandas(latest_data)

latest_cudf['month'] = latest_cudf['date'].dt.month
latest_cudf['day_of_week'] = latest_cudf['date'].dt.weekday
latest_cudf['mandi_code'] = latest_cudf['mandi_id'].astype('category').cat.codes
latest_cudf['commodity_code'] = latest_cudf['commodity_id'].astype('category').cat.codes

d_latest = xgb.DMatrix(latest_cudf[features])
predictions = model.predict(d_latest)

# Create the forecast DataFrame
forecast_pd = latest_data[['mandi_id', 'commodity_id', 'date']].copy()
forecast_pd['forecast_date'] = forecast_pd['date'] + pd.Timedelta(days=7)
forecast_pd['predicted_price'] = predictions.get() # Convert cupy back to numpy
forecast_pd['model_version'] = 'xgboost_v1'
forecast_pd['created_at'] = pd.Timestamp.utcnow()

# Keep only BQ schema columns
forecast_pd = forecast_pd[['mandi_id', 'commodity_id', 'forecast_date', 'predicted_price', 'model_version', 'created_at']]

print("📤 Uploading forecasts to BigQuery...")
job_config = bigquery.LoadJobConfig(write_disposition="WRITE_APPEND")
job = client.load_table_from_dataframe(forecast_pd, f"{PROJECT_ID}.{DATASET_ID}.fact_forecast", job_config=job_config)
job.result()

print("✅ Forecasting Phase Complete! Predictions are live in the database.")